In [23]:
import pandas as pd

# Load the raw data
df = pd.read_csv('mp_routes.csv')

display(df.head())
df.shape

,Unnamed: 0,Route,Location,URL,Avg Stars,Route Type,Rating,Pitches,Length,Area Latitude,Area Longitude,desc,protection,num_votes
0,0,Access Denied,El Mirador > El Potrero Chico > Nuevo Leon > N...,https://www.mountainproject.com/route/11014983...,2.9,Sport,5.10b/c,4,350.0,25.95044,-100.47755,This is a really great route~ with awesome exp...,12 draws + 60m Rope Take 22 draws if you wan...,22
1,1,Agave Nectar,Sugar Shack > Cougar Canyon (Creek) - CONSTRUC...,https://www.mountainproject.com/route/11091386...,2.0,Sport,5.10b/c,1,NaN,51.09642,-115.31767,from tabvar: Cool fins to roof~ thin holds...,4 bolts to anchor,1
2,2,Ant & Bee do Yoga,The Hen House > Kamloops > British Columbia > ...,https://www.mountainproject.com/route/11240652...,2.7,Trad,5.10b/c,1,NaN,50.57212,-120.13874,A safe mixed route with a bit of run out up to...,"mixed~ gear to 4""",3
3,3,Besame Fuerte,Pilon De Lolita > Loreto Area > Baja Californi...,https://www.mountainproject.com/route/11608640...,2.0,Sport,5.10b/c,1,80.0,26.01097,-111.34166,Start on a slab under a left leaning arched ro...,bolts,1
4,4,Big Momma's Rock,The Courtyard > Mamquam FSR > Squamish > Briti...,https://www.mountainproject.com/route/11445772...,3.0,Sport,5.10b/c,1,60.0,49.71393,-123.09943,Fun technical climbing. Tricky right off the bat.,bolts,3


(116700, 14)

In [1]:
import pandas as pd
import numpy as np
import torch

# Load the initial dataset
df = pd.read_csv('mp_routes.csv')

# Clean and normalize column names for consistency
if 'Unnamed: 0' in df.columns:
    df = df.rename(columns={'Unnamed: 0': 'route_id'})
else:
    df.insert(0, 'route_id', range(len(df)))

df.columns = [c.strip().lower().replace(' ', '_').replace('.', '') for c in df.columns]

print(f"Original dataset size: {df.shape[0]} routes")

# Filter the dataset to include only California routes
df = df[df['location'].str.contains('California', na=False)].copy()

# Define and filter for a specific set of target difficulty grades
target_grades = [
    "5.7", "5.8", "5.9", "5.10a", "5.10b", "5.10c", "5.10d", 
    "5.11a", "5.11b", "5.11c", "5.11d", "5.12a", "5.12b"
]

# Keep only the routes matching our specific grade list
df = df[df['rating'].isin(target_grades)].copy()

print(f"Filtered dataset size (California + specific grades): {df.shape[0]} routes")
print(f"Missing values in length column: {df['length'].isna().sum()}")
print(f"Median route length: {df['length'].median():.2f}m")

# Map categorical grades to numerical values (0-12) for the model
grade_map = {grade: i for i, grade in enumerate(target_grades)}
df['rating_num'] = df['rating'].map(grade_map)

# Process the location hierarchy to build nodes and edges for the graph
location_nodes = set()
edges = []

for idx, row in df.iterrows():
    if pd.isna(row['location']): 
        continue
        
    parts = [p.strip() for p in str(row['location']).split('>')]
    route_node_id = f"route_{row['route_id']}"
    
    # Establish the route-sector relationship
    edges.append({'source': route_node_id, 'target': parts[0], 'type': 'IN_AREA'})
    
    # Establish parent-child relationships within the area hierarchy
    for i in range(len(parts) - 1):
        location_nodes.add(parts[i])
        location_nodes.add(parts[i+1])
        edges.append({'source': parts[i], 'target': parts[i+1], 'type': 'SUB_AREA_OF'})

# Save the processed data into separate CSV files for Neo4j import
df[['route_id', 'route', 'avg_stars', 'rating', 'rating_num', 'pitches', 
    'length','route_type', 'area_latitude', 'area_longitude']].to_csv('neo4j_routes_california.csv', index=False)
pd.DataFrame({'name': list(location_nodes)}).to_csv('neo4j_locations_california.csv', index=False)
pd.DataFrame(edges).drop_duplicates().to_csv('neo4j_edges_california.csv', index=False)

# Prepare the data structure and edge index for the PyTorch Geometric GNN
all_entities = pd.concat([df['route_id'].apply(lambda x: f"route_{x}"), pd.Series(list(location_nodes))]).unique()
mapping = {name: i for i, name in enumerate(all_entities)}

edge_index = torch.tensor([
    [mapping[e['source']] for e in edges],
    [mapping[e['target']] for e in edges]
], dtype=torch.long)

print(f"\nTotal nodes: {len(all_entities)} | Total edges: {edge_index.shape[1]}")
print(f"Difficulty mapping (0-12):")
for g, n in grade_map.items():
    print(f"  {n}: {g}")

Original dataset size: 116700 routes
Filtered dataset size (California + specific grades): 7302 routes
Missing values in length column: 577
Median route length: 65.00m

Total nodes: 10405 | Total edges: 38222
Difficulty mapping (0-12):
  0: 5.7
  1: 5.8
  2: 5.9
  3: 5.10a
  4: 5.10b
  5: 5.10c
  6: 5.10d
  7: 5.11a
  8: 5.11b
  9: 5.11c
  10: 5.11d
  11: 5.12a
  12: 5.12b
